# Bronze Ingestion — Raw Data Loading & Traceability

## Purpose

This notebook ingests raw Auto Insurance Policies and Claims datasets from CSV files stored in a Databricks Unity Catalog Volume into Delta tables in the Bronze layer.

The main objective is to preserve source data while adding technical metadata to support traceability, auditing, and downstream processing.

## Why is this important?

- **Data Ingestion:** Load raw Policies and Claims data from CSV files into Databricks using PySpark.
- **Raw Data Preservation:** Store source data in Delta format without applying business transformations.
- **Data Traceability:** Track each record using ingestion metadata such as `batch_id`, `source_file`, and `ingestion_timestamp`.
- **Schema Inspection:** Validate source file structures, column names, and inferred data types.
- **Data Auditing:** Verify record counts to ensure successful ingestion.
- **Pipeline Foundation:** Establish a reliable data source for Silver-layer transformations and data quality checks.

## Expected Outcome

The notebook creates two Delta tables:

- `auto_insurance.bronze.policies_raw` — Raw policy records, including driver, vehicle, exposure, and claim count information.
- `auto_insurance.bronze.claims_raw` — Raw claim records containing policy identifiers and claim amounts.

Both tables include ingestion metadata to support monitoring, troubleshooting, and future incremental processing.

**Note:** This notebook uses overwrite mode for the initial full-load ingestion. Incremental ingestion, schema enforcement, and automated ingestion monitoring will be introduced in future iterations.

In [0]:
# Define the Unity Catalog location containing the source CSV files.
# Keeping the paths configurable makes the ingestion notebook easier to maintain.

catalog = "auto_insurance"
schema = "bronze"
volume = "source_files"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"

policies_path = f"{base_path}/freMTPL2freq.csv"
claims_path = f"{base_path}/freMTPL2sev.csv"

display(dbutils.fs.ls(base_path))

In [0]:
# Read both CSV datasets into Spark DataFrames.
# The header option uses the first CSV row as column names.
# Schema inference determines data types from the source data; production pipelines
# may prefer explicit schemas to prevent unexpected type changes.

df_policies = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(policies_path)
)

df_claims = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(claims_path)
)

In [0]:
# Inspect record counts, inferred schemas, and sample records before ingestion.
# These checks provide a quick baseline for troubleshooting source data issues.

print("Policies count:", df_policies.count())
print("Claims count:", df_claims.count())

df_policies.printSchema()
df_claims.printSchema()

display(df_policies.limit(10))
display(df_claims.limit(10))

In [0]:
# Generate one unique batch identifier for this notebook execution.
# Add ingestion metadata to both datasets without changing source business columns.
# The timestamp records ingestion time, source_file identifies the originating CSV,
# and batch_id links both datasets to the same ingestion run.

from pyspark.sql import functions as F
import uuid

batch_id = str(uuid.uuid4())

df_policies_bronze = (
    df_policies
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("freMTPL2freq.csv"))
    .withColumn("batch_id", F.lit(batch_id))
)

df_claims_bronze = (
    df_claims
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("freMTPL2sev.csv"))
    .withColumn("batch_id", F.lit(batch_id))
)

In [0]:
# Persist the raw datasets as managed Delta tables in the Bronze schema.
# Overwrite implements full-snapshot ingestion: every run replaces table contents.
# This is not incremental loading; Delta table history can track write operations,
# but newly arrived records are not identified separately by this notebook.

(
    df_policies_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("auto_insurance.bronze.policies_raw")
)

(
    df_claims_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("auto_insurance.bronze.claims_raw")
)